# Pré-processamento 2 - Base de palavras

## Carregando os dados

In [1]:
# importa as bibliotecas usadas no notebook
# importando bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pandas.plotting import scatter_matrix

# importa o recurso Path para indicar o caminho do arquivo
# importando o arquivo
from pathlib import Path

df = pd.read_excel(
    Path("Corpus SANTOS TONI_FDC.xlsx"),
    sheet_name="Plan2"
)

df.head()

,PALAVRA,LEMA,CATMORF,TRANSCRICAO,ACENTUACAO,ESTSILABICA,CATACENTUAL,FREQ
0,que,que,Conjunção Subordinada,&ke%,&ke%,&CV%,mono,22627
1,é,ser,Verbo Principal Indicativo Presente Terceira P...,&5%,&5%,&V%,oxítona,17108
2,o,o,Determinante Artigo Masculino Singular,&o%,&o%,&V%,mono,16513
3,não,não,Advérbio Negativo,&nAW%,&n2W%,&CVG%,oxítona,11696
4,a,o,Determinante Artigo Feminino Singular,&a%,&a%,&V%,mono,10546


## Verificação das duplicidades

In [2]:
# conta quantas linhas são completamente iguais em todas as colunas
# o resultado indica quantas duplicidades existem antes da remoção
df.duplicated().sum()

np.int64(138)

In [3]:
# mostra todas as linhas que fazem parte de algum grupo duplicado
# keep=False mantém todas as ocorrências, e não somente a repetida
df[df.duplicated(keep=False)]

,PALAVRA,LEMA,CATMORF,TRANSCRICAO,ACENTUACAO,ESTSILABICA,CATACENTUAL,FREQ
2954,Ben,ben,Nome Própio Pessoa,&bEJ%,&b6J%,&CVG%,oxítona,5
2955,Ben,ben,Nome Própio Pessoa,&bEJ%,&b6J%,&CVG%,oxítona,5
4063,depois,depois,Advérbio Geral,&de-poJs%,&de-p9Js%,&CV-CVGS%,oxítona,3
4064,depois,depois,Advérbio Geral,&de-poJs%,&de-p9Js%,&CV-CVGS%,oxítona,3
4245,índia,índio,Nome Comun Feminino Singular,&8-DJa%,&8-DJ@%,&V-CGV%,paroxítona,3
...,...,...,...,...,...,...,...,...
11531,vira,ver,Verbo Principal Indicativo Primeira Pessoa Sin...,&vi-ra%,&v7-r@%,&CV-CV%,paroxítona,1
11544,visitando,visitar,Verbo Principal Gerúndio,&vi-zi-tA-do%,&vi-zi-t2-dw%,&CV-CV-CV-CV%,paroxítona,1
11545,visitando,visitar,Verbo Principal Gerúndio,&vi-zi-tA-do%,&vi-zi-t2-dw%,&CV-CV-CV-CV%,paroxítona,1
11556,viveu,viver,Verbo Principal Indicativo Passado Terceira Pe...,&vi-veW%,&vi-v4W%,&CV-CVG%,oxítona,1


## Remoção das duplicidades completas

In [4]:
# remove as linhas completamente duplicadas
# apenas uma ocorrência de cada linha igual é mantida
df = df.drop_duplicates()

# mostra o novo tamanho da base
df.shape

(11504, 8)

In [5]:
# seleciona somente as colunas que armazenam textos
colunas_texto = df.select_dtypes(include='object').columns

# verifica espaços no início ou no final de cada valor textual
for coluna in colunas_texto:
    espacos = (df[coluna] != df[coluna].str.strip()).sum()
    print(coluna, ":", espacos)

PALAVRA : 0
LEMA : 37
CATMORF : 0
TRANSCRICAO : 11503
ACENTUACAO : 11503
ESTSILABICA : 11504
CATACENTUAL : 11489


In [6]:
# remove espaços no início e no final dos valores textuais
# espaços existentes entre palavras não são removidos
for coluna in colunas_texto:
    df[coluna] = df[coluna].str.strip()

In [7]:
# verifica novamente se ainda existem espaços extras
for coluna in colunas_texto:
    espacos = (df[coluna] != df[coluna].str.strip()).sum()
    print(coluna, ":", espacos)

PALAVRA : 0
LEMA : 37
CATMORF : 0
TRANSCRICAO : 0
ACENTUACAO : 0
ESTSILABICA : 0
CATACENTUAL : 54


In [8]:
# investiga possíveis espaços extras nas colunas analisadas
# repete a verificação ignorando células vazias (NaN)
for coluna in ['LEMA', 'CATACENTUAL']:
    # mostra o nome da coluna analisada
    # mostra a quantidade encontrada na coluna
    print('\nColuna:', coluna)
    # seleciona valores diferentes após retirar espaços das extremidades
    # seleciona somente valores preenchidos com espaços extras
    valores = df.loc[
        df[coluna] != df[coluna].str.strip(),
        coluna
    ].drop_duplicates()

    # exibe os valores encontrados, preservando caracteres invisíveis
    for valor in valores:
        print(repr(valor))


Coluna: LEMA
nan

Coluna: CATACENTUAL
nan


In [9]:
for coluna in ['LEMA', 'CATACENTUAL']:
    valores = df.loc[
        df[coluna].notna() &
        (df[coluna] != df[coluna].str.strip()),
        coluna
    ].drop_duplicates()

    print('\nColuna:', coluna)
    print('Quantidade:', len(valores))

    for valor in valores:
        print(repr(valor))


Coluna: LEMA
Quantidade: 0

Coluna: CATACENTUAL
Quantidade: 0


In [10]:
# conta os valores ausentes em cada coluna após os tratamentos
df.isnull().sum()

PALAVRA         0
LEMA           37
CATMORF         0
TRANSCRICAO     0
ACENTUACAO      0
ESTSILABICA     0
CATACENTUAL    54
FREQ            0
dtype: int64

In [11]:
df[df['LEMA'].isnull()][
    ['PALAVRA', 'LEMA', 'CATACENTUAL']
].head(10)

,PALAVRA,LEMA,CATACENTUAL
115,dois,NaN,oxítona
246,duas,NaN,oxítona
272,quatro,NaN,paroxítona
305,cinco,NaN,paroxítona
471,sete,NaN,paroxítona
568,seis,NaN,oxítona
695,vinte,NaN,paroxítona
712,oito,NaN,paroxítona
733,dez,NaN,oxítona
782,nove,NaN,paroxítona


In [12]:
df[df['CATACENTUAL'].isnull()][
    ['PALAVRA', 'LEMA', 'CATACENTUAL']
].head(10)

,PALAVRA,LEMA,CATACENTUAL
1400,helicóptero,helicóptero,NaN
3379,ChapéuzinhoVermelho,chapéuzinhovermelho,NaN
3747,tricerápodesquevoa,tricerápodesquevoa,NaN
3840,amígdala,amigo,NaN
4014,cocócocócocócocócocó,cocócocócocócocócocó,NaN
4380,MuseudeHistóriaNatural,museudehistórianatural,NaN
4498,pterodáctilo,pterodáctilo,NaN
4735,aHistóriadoovoestranho,ahistóriadoovoestranho,NaN
5031,cócócócócó,cócócócócó,NaN
5307,estáchorando,estáchorando,NaN


In [13]:
df_original = pd.read_excel(
    Path("Corpus SANTOS TONI_FDC.xlsx"),
    sheet_name="Plan2"
)

df_original.isnull().sum()

PALAVRA        0
LEMA           0
CATMORF        0
TRANSCRICAO    0
ACENTUACAO     0
ESTSILABICA    0
CATACENTUAL    0
FREQ           0
dtype: int64

In [14]:
# padroniza para letras minúsculas as colunas de texto comum
# preserva as letras maiúsculas das colunas que contêm códigos técnicos
df['PALAVRA'] = df['PALAVRA'].str.lower()
df['LEMA'] = df['LEMA'].str.lower()
df['CATACENTUAL'] = df['CATACENTUAL'].str.lower()

In [15]:
# exibe as primeiras linhas para conferir a padronização
df.head()

,PALAVRA,LEMA,CATMORF,TRANSCRICAO,ACENTUACAO,ESTSILABICA,CATACENTUAL,FREQ
0,que,que,Conjunção Subordinada,&ke%,&ke%,&CV%,mono,22627
1,é,ser,Verbo Principal Indicativo Presente Terceira P...,&5%,&5%,&V%,oxítona,17108
2,o,o,Determinante Artigo Masculino Singular,&o%,&o%,&V%,mono,16513
3,não,não,Advérbio Negativo,&nAW%,&n2W%,&CVG%,oxítona,11696
4,a,o,Determinante Artigo Feminino Singular,&a%,&a%,&V%,mono,10546


In [16]:
# procura o caractere de substituição nas colunas de texto
# esse caractere pode indicar problema de codificação dos dados
for coluna in colunas_texto:
    quantidade = df[coluna].astype(str).str.contains('�', na=False).sum()
    print(coluna, ':', quantidade)

PALAVRA : 0
LEMA : 0
CATMORF : 0
TRANSCRICAO : 0
ACENTUACAO : 0
ESTSILABICA : 0
CATACENTUAL : 0


In [17]:
# exibe exemplos de palavras para observar a grafia e os acentos
df['PALAVRA'].head(20)

0       que
1         é
2         o
3       não
4         a
5      vocé
6      está
7      olha
8     estou
9      aqui
10      vai
11       de
12        e
13     para
14      tem
15       um
16       aí
17     esse
18    vamos
19      ele
Name: PALAVRA, dtype: object

In [18]:
# seleciona exemplos de palavras que possuem acentos ou cedilha
# os acentos legítimos são preservados e não são removidos
df[df['PALAVRA'].str.contains('[áàãâéêíóôõúç]', case=False, na=False)].head(20)

,PALAVRA,LEMA,CATMORF,TRANSCRICAO,ACENTUACAO,ESTSILABICA,CATACENTUAL,FREQ
1,é,ser,Verbo Principal Indicativo Presente Terceira P...,&5%,&5%,&V%,oxítona,17108
3,não,não,Advérbio Negativo,&nAW%,&n2W%,&CVG%,oxítona,11696
5,vocé,vocé,Pronome Pessoal Terceira Pessoa Singular,&vo-k5%,&vo-k5%,&CV-CV%,oxítona,10178
6,está,estar,Verbo Principal Indicativo Presente Terceira P...,&es-t1%,&es-t1%,&VC-CV%,oxítona,7761
16,aí,aí,Advérbio Geral,&a-7%,&a-7%,&V-V%,oxítona,3734
26,lá,o,Advérbio Geral,&la%,&l1%,&CV%,oxítona,2187
31,então,então,Advérbio Geral,&E-tAW%,&E-t2W%,&V-CVG%,oxítona,2048
35,mamãe,mamãe,Nome Comun Feminino Singular,&ma-mA-e%,&ma-m2-y%,&CV-CV-V%,paroxítona,1846
42,né,né,Nome Própio Pessoa,&n5%,&n5%,&CV%,oxítona,1614
46,cadé,cadé,Verbo Principal Indicativo Presente Terceira P...,&ka-d5%,&ka-d5%,&CV-CV%,oxítona,1432


## Salvando a cópia atual da base

In [19]:
# salva a versão atual tratada em um novo arquivo
# o arquivo original permanece preservado
df.to_excel("Corpus_SANTOS_TONI_FDC_tratado.xlsx", index=False)